In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

# 1. Өгөгдөл унших
DATA_PATH = "output.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# 2. Өгөгдөл шүүх (Зөвхөн давж заалдах болон хяналтын шатны хэргүүд)
model_df = df[df["case_level"].isin(["appellate", "supervisory"])].copy()
model_df = model_df[model_df["appeal_outcome"].notna()].copy()

# 1 = хэвээр үлдээсэн (upheld), 0 = бусад (өөрчилсөн, хүчингүй болгосон)
model_df["шийдвэр_хэвээр"] = (model_df["appeal_outcome"] == "upheld").astype(int)

# Зөвхөн үнийн дүнтэй хэргүүдийг ялгаж авах (Smoothed Regression хийхийн тулд)
reg_df = model_df[model_df["claim_amount_requested"].notna()].copy()

# Үнийн дүнг Log scale-рүү хөрвүүлэх (хэт их зөрүүг багасгахын тулд)
reg_df["log_claim"] = np.log1p(reg_df["claim_amount_requested"])

# 3. Smoothed Regression (LOWESS) Анализ
# x = Нэхэмжлэлийн дүн (log), y = Шийдвэр хэвээр үлдсэн эсэх (0 эсвэл 1)
x = reg_df["log_claim"].values
y = reg_df["шийдвэр_хэвээр"].values

print(f"Шинжилгээнд орсон хэргийн тоо: {len(x)}")

# LOWESS (Locally Weighted Scatterplot Smoothing) регресс ажиллуулах
# frac = 0.3 гэдэг нь цэг бүрийг тооцоолохдоо нийт өгөгдлийн 30%-ийг харгалзан үзнэ гэсэн үг
lowess = sm.nonparametric.lowess(y, x, frac=0.3)

# Үр дүнг салгаж авах
lowess_x = lowess[:, 0]
lowess_y = lowess[:, 1]

# 4. График байгуулах (Visual Analysis)
plt.figure(figsize=(12, 6))

# Бодит цэгүүдийг зурах (0 болон 1 дээр байгаа)
sns.scatterplot(x=x, y=y, alpha=0.1, color="gray", label="Бодит хэргүүд (1=Хэвээр, 0=Өөрчилсөн)")

# Smoothed Regression шугамыг зурах
plt.plot(lowess_x, lowess_y, color="red", linewidth=3, label="Smoothed Regression (LOWESS) Шугам")

# Графикийн тохиргоо
plt.title("Нэхэмжлэлийн үнийн дүн болон Шүүхийн шийдвэр хэвээр үлдэх магадлал (Smoothed Regression)", fontsize=14)
plt.xlabel("Нэхэмжлэлийн дүн (Log Scale)", fontsize=12)
plt.ylabel("Шийдвэр хэвээр үлдэх магадлал", fontsize=12)
plt.ylim(-0.1, 1.1)
plt.axhline(y=reg_df["шийдвэр_хэвээр"].mean(), color="blue", linestyle="--", alpha=0.5, label="Дундаж магадлал")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'output.csv'

In [14]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Өгөгдөл унших (Load Data)
df = pd.read_csv("output.csv", encoding="utf-8-sig")
total_rows = len(df)

print("ДАТА БҮТЦИЙН БОЛОН ЧАНАР\n" + "="*60)
print(f"Нийт хэргийн тоо (Total Rows): {total_rows}")

# 1. Target Variable Completeness (Шийдвэрийн үр дүн хэр сайн олдсон бэ?)
first_inst = df[df['case_level'] == 'first_instance']
appeals = df[df['case_level'].isin(['appellate', 'supervisory'])]

first_inst_fill = first_inst['civil_outcome'].notna().mean() * 100
appeal_fill = appeals['appeal_outcome'].notna().mean() * 100

print("\n1. TARGET VARIABLES:")
print(f" - Анхан шатны шийдвэр (civil_outcome) олдсон хувь: {first_inst_fill:.1f}%")
print(f" - Давж/Хяналтын шийдвэр (appeal_outcome) олдсон хувь: {appeal_fill:.1f}%")

# 2. Feature Completeness (Оролтын шинж чанарууд)
print("\n2. STRUCTURED FEATURES:")
features = ['court', 'dispute_type', 'plaintiff_type', 'defendant_type', 'claim_amount_requested']
for f in features:
    fill_rate = df[f].notna().mean() * 100
    print(f" - '{f}' баганын дүүргэлт: {fill_rate:.1f}%")

# 3. Data Leakage Check (Текст дотор шүүгчийн шийдвэр орсон эсэхийг шалгах)
# Хэрвээ case_narrative дотор "ТОГТООХ нь:" эсвэл "ҮНДЭСЛЭХ нь:" байвал алдаатай (Leakage) гэсэн үг.
print("\n3. DATA LEAKAGE CHECK:")
narratives = df['case_narrative'].dropna()

leaked_outcome = narratives.str.contains("ТОГТООХ нь|ЗАХИРАМЖЛАХ нь", case=False).sum()
leaked_reasoning = narratives.str.contains("ҮНДЭСЛЭХ нь", case=False).sum()

print(f" - Нийт тексттэй хэрэг: {len(narratives)}")
print(f" - Шийдвэрлэх хэсэг (ТОГТООХ нь) алдагдсан хэрэг: {leaked_outcome}")
print(f" - Шүүгчийн үндэслэл (ҮНДЭСЛЭХ нь) алдагдсан хэрэг: {leaked_reasoning}")

if leaked_outcome == 0 and leaked_reasoning == 0:
    print("\n✅ ДҮГНЭЛТ: Текст маш цэвэр байна! Data Leakage байхгүй тул NLP модел сургахад бэлэн.")
else:
    print("\n⚠️ ДҮГНЭЛТ: Зарим текстэд Data Leakage байна. Үүнийг цэвэрлэх шаардлагатай.")


ДАТА БҮТЦИЙН БОЛОН ЧАНАР
Нийт хэргийн тоо (Total Rows): 1168

1. TARGET VARIABLES:
 - Анхан шатны шийдвэр (civil_outcome) олдсон хувь: 100.0%
 - Давж/Хяналтын шийдвэр (appeal_outcome) олдсон хувь: 98.4%

2. STRUCTURED FEATURES:
 - 'court' баганын дүүргэлт: 100.0%
 - 'dispute_type' баганын дүүргэлт: 100.0%
 - 'plaintiff_type' баганын дүүргэлт: 97.7%
 - 'defendant_type' баганын дүүргэлт: 96.4%
 - 'claim_amount_requested' баганын дүүргэлт: 78.1%

3. DATA LEAKAGE CHECK:
 - Нийт тексттэй хэрэг: 648
 - Шийдвэрлэх хэсэг (ТОГТООХ нь) алдагдсан хэрэг: 0
 - Шүүгчийн үндэслэл (ҮНДЭСЛЭХ нь) алдагдсан хэрэг: 0

✅ ДҮГНЭЛТ: Текст маш цэвэр байна! Data Leakage байхгүй тул NLP модел сургахад бэлэн.
